# 🎙️ Whisper AI 語音辨識工具

> 基於 OpenAI Whisper 的語音辨識 / 字幕生成 Colab 工具

## 主要功能
| 類別 | 功能 |
|---|---|
| 影音來源 | YouTube URL 自動下載 ・ 本機上傳（mp3/mp4/m4a/wav/mov…）|
| 語言支援 | 中 / 英 / 日 / 韓 / 西 / 法 / 德 / 俄… **14+ 語言**，或自動偵測 |
| 辨識模型 | tiny → large-v3 共 6 種，自由切換準確度與速度 |
| 翻譯模式 | 將任意語言翻譯成英文（task = translate）|
| 輸出格式 | TXT ・ SRT ・ VTT ・ JSON ・ TSV（**五種格式**）|
| GPU 加速 | 自動偵測並使用 GPU |
| 雲端整合 | 儲存至 Google Drive |

## 使用流程
1. **STEP 1** 安裝套件 + 檢查環境
2. **STEP 2** 設定模型 / 語言 / 進階選項
3. **STEP 3** 提供 YouTube URL 或上傳檔案
4. **STEP 4** 執行辨識
5. **STEP 5** 檢視結果
6. **STEP 6** 匯出與下載

> 💡 **建議**：執行階段 → 變更執行階段類型 → 選擇 **T4 GPU**（免費）以加速辨識。

---


## STEP 1：環境設定


In [ ]:
#@title 1-1 安裝必要套件 { display-mode: "form" }
#@markdown 安裝 `openai-whisper`、`yt-dlp`、`ffmpeg`，僅需執行一次

!pip install -q -U openai-whisper
!pip install -q -U yt-dlp
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

print("所有套件安裝完成！")


In [ ]:
#@title 1-2 環境檢查 { display-mode: "form" }
#@markdown 確認是否有可用 GPU（強烈建議啟用）

import torch, platform, sys

print("━" * 40)
print(f"Python      : {sys.version.split()[0]}")
print(f"PyTorch     : {torch.__version__}")
print(f"OS          : {platform.platform()}")
print("━" * 40)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU 可用    : {gpu_name}")
    print(f"VRAM        : {gpu_mem:.1f} GB")
    print("⚡ 將使用 GPU 加速辨識")
else:
    print("未偵測到 GPU，將使用 CPU（速度較慢）")
    print("啟用方式：選單 → 執行階段 → 變更執行階段類型 → T4 GPU")
print("━" * 40)


## STEP 2：辨識設定

調整模型大小、目標語言、與進階選項。


In [ ]:
#@title 2-1 模型與語言設定 { display-mode: "form" }

#@markdown ### 模型大小
#@markdown 越大越準，但下載與運算時間越長
model_size = "large-v3" #@param ["tiny", "base", "small", "medium", "large-v2", "large-v3"]

#@markdown ### 辨識語言（選 `auto` 由模型自動偵測）
language = "zh" #@param ["auto", "zh", "en", "ja", "ko", "es", "fr", "de", "ru", "pt", "it", "ar", "th", "vi", "id"]

#@markdown ### 任務類型
#@markdown - `transcribe`：辨識原文
#@markdown - `translate`：將任何語言翻譯成英文
task = "transcribe" #@param ["transcribe", "translate"]

#@markdown ---
#@markdown ### 進階選項

#@markdown 提示詞（引導辨識：可指定專有名詞、標點風格）
initial_prompt = "" #@param {type:"string"}

#@markdown Beam Size（搜尋寬度：越大越準但越慢）
beam_size = 5 #@param {type:"slider", min:1, max:10, step:1}

#@markdown 是否輸出字級時間戳（部分後處理需要）
word_timestamps = False #@param {type:"boolean"}

#@markdown 是否過濾低品質片段（建議勾選）
filter_low_quality = True #@param {type:"boolean"}

# 儲存全域設定
CONFIG = dict(
    model_size=model_size, language=language, task=task,
    initial_prompt=initial_prompt, beam_size=beam_size,
    word_timestamps=word_timestamps, filter_low_quality=filter_low_quality,
)

print("目前設定")
print("━" * 40)
print(f"  模型      : {model_size}")
print(f"  語言      : {language if language != 'auto' else '自動偵測'}")
print(f"  任務      : {'辨識原文' if task == 'transcribe' else '翻譯成英文'}")
print(f"  Beam Size : {beam_size}")
print(f"  字級時戳  : {'開' if word_timestamps else '關'}")
print(f"  品質過濾  : {'開' if filter_low_quality else '關'}")
if initial_prompt:
    print(f"  提示詞    : {initial_prompt}")
print("━" * 40)


## STEP 3：選擇影音來源

**擇一**：填入 YouTube URL，或勾選使用本機上傳。


In [ ]:
#@title 3-1 YouTube 影片資訊預覽 { display-mode: "form" }
#@markdown 填入 YouTube URL，先確認影片資訊（不下載）

youtube_url = "https://www.youtube.com/watch?v=qXwt67lyhsM" #@param {type:"string"}

import yt_dlp

VIDEO_INFO = None
if youtube_url.strip():
    with yt_dlp.YoutubeDL({'quiet': True, 'skip_download': True, 'no_warnings': True}) as ydl:
        VIDEO_INFO = ydl.extract_info(youtube_url, download=False)
    dur = VIDEO_INFO.get('duration', 0)
    print("影片資訊")
    print("━" * 40)
    print(f"  標題  : {VIDEO_INFO.get('title')}")
    print(f"  作者  : {VIDEO_INFO.get('uploader')}")
    print(f"  長度  : {dur // 60:.0f} 分 {dur % 60:.0f} 秒")
    print(f"  觀看  : {VIDEO_INFO.get('view_count', 0):,}")
    print("━" * 40)
else:
    print("ℹ未填入 YouTube URL（將使用本機上傳模式）")


In [ ]:
#@title 3-2 下載 / 上傳音訊檔 { display-mode: "form" }
#@markdown - 若 STEP 3-1 有填 URL，會自動下載 YouTube 音訊
#@markdown - 否則勾選下方選項，從本機上傳檔案

use_local_upload = False #@param {type:"boolean"}

import os, subprocess, re

audio_file = None
base_name = "output"

def safe_name(s, n=60):
    s = re.sub(r'[\\/*?:"<>|]', '_', s).strip()
    return s[:n] if s else "output"

if youtube_url.strip() and not use_local_upload:
    title = VIDEO_INFO.get('title', 'youtube_audio') if VIDEO_INFO else 'youtube_audio'
    base_name = safe_name(title)
    ydl_opts = {
        'format': 'bestaudio/best',
        'outtmpl': 'input_audio.%(ext)s',
        'overwrites': True,
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'm4a',
            'preferredquality': '192',
        }],
        'quiet': True,
        'no_warnings': True,
    }
    print("下載中...")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])
    audio_file = "input_audio.m4a"
    print(f"下載完成")
elif use_local_upload:
    from google.colab import files
    print("📁 請選擇要上傳的檔案（支援 mp3/mp4/m4a/wav/mov/mkv 等）")
    uploaded = files.upload()
    filename = list(uploaded.keys())[0]
    base_name = safe_name(os.path.splitext(filename)[0])
    audio_file = f"input_audio.mp3"
    print("轉換為標準音訊格式...")
    subprocess.run(
        ['ffmpeg', '-y', '-i', filename, '-vn', '-acodec', 'libmp3lame', '-ar', '16000', audio_file],
        check=True, capture_output=True
    )
    print("音訊準備完成")
else:
    raise RuntimeError("請於 3-1 填入 YouTube URL，或勾選『使用本機上傳』後重新執行")

size_mb = os.path.getsize(audio_file) / 1e6
print("━" * 40)
print(f"  檔案    : {audio_file}")
print(f"  大小    : {size_mb:.2f} MB")
print(f"  輸出名稱: {base_name}")
print("━" * 40)


## STEP 4：執行語音辨識


In [ ]:
#@title 4-1 載入模型 & 開始辨識 { display-mode: "form" }
import whisper, time, torch, os

assert audio_file and os.path.exists(audio_file), "找不到音訊檔，請先執行 STEP 3"

# 載入模型
print(f"載入模型：{CONFIG['model_size']}")
print("   首次使用會下載權重，依模型大小需數秒到數分鐘")
t0 = time.time()
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model(CONFIG['model_size'], device=device)
print(f"模型載入完成（{time.time()-t0:.1f}s，裝置：{device}）\n")

# 組合辨識選項
options = dict(
    task=CONFIG['task'],
    beam_size=CONFIG['beam_size'],
    word_timestamps=CONFIG['word_timestamps'],
    verbose=False,
    fp16=(device == "cuda"),
)
if CONFIG['language'] != "auto":
    options['language'] = CONFIG['language']
if CONFIG['initial_prompt']:
    options['initial_prompt'] = CONFIG['initial_prompt']

# 執行辨識
print("開始辨識...")
t0 = time.time()
result = model.transcribe(audio_file, **options)
elapsed = time.time() - t0

print("━" * 40)
print(f"辨識完成")
print(f"  ⏱耗時      : {elapsed:.1f} 秒")
print(f"  偵測語言  : {result.get('language')}")
print(f"  段落數    : {len(result['segments'])}")
print(f"  總字數    : {len(result['text'])}")
print("━" * 40)


## STEP 5：辨識結果


In [ ]:
#@title 5-1 顯示完整文本 { display-mode: "form" }
from IPython.display import HTML, display
from html import escape

text = result["text"].strip()
char_count = len(text)
# 對 CJK 友善的「詞」估算：以非 CJK 連續字元為一詞，CJK 每字一詞
import re as _re
_cjk = len(_re.findall(r"[\u4e00-\u9fff\u3040-\u30ff\uac00-\ud7af]", text))
_non = len(_re.findall(r"[A-Za-z\u00C0-\u024F]+", text))
word_count = _cjk + _non

html = f'''
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif;">
  <div style="background:linear-gradient(90deg,#667eea,#764ba2);color:#fff;
              padding:12px 18px;border-radius:8px 8px 0 0;
              display:flex;justify-content:space-between;align-items:center;">
    <strong>辨識結果</strong>
    <span style="font-size:13px;opacity:.9">{char_count} 字 ・ {word_count} 詞</span>
  </div>
  <div style="padding:18px;background:#fafafa;border:1px solid #e0e0e0;
              border-top:none;border-radius:0 0 8px 8px;
              line-height:1.8;font-size:15px;white-space:pre-wrap;
              max-height:400px;overflow-y:auto;">{escape(text)}</div>
</div>
'''
display(HTML(html))


In [ ]:
#@title 5-2 逐段時間軸 { display-mode: "form" }
#@markdown 顯示每段的時間戳記與文字

from IPython.display import HTML, display
from html import escape

def fmt(s):
    m, s = divmod(s, 60); h, m = divmod(m, 60)
    return f"{int(h):02d}:{int(m):02d}:{s:05.2f}"

rows = []
for i, seg in enumerate(result["segments"], 1):
    lp = seg.get("avg_logprob", 0)
    conf = "🟢" if lp > -0.5 else ("🟡" if lp > -1.0 else "🔴")
    rows.append(
        f'<tr><td style="padding:6px 10px;color:#888;font-family:monospace">{i}</td>'
        f'<td style="padding:6px 10px;text-align:center">{conf}</td>'
        f'<td style="padding:6px 10px;font-family:monospace;color:#667eea;white-space:nowrap">{fmt(seg["start"])} → {fmt(seg["end"])}</td>'
        f'<td style="padding:6px 10px;">{escape(seg["text"].strip())}</td></tr>'
    )

html = f'''
<div style="font-family:-apple-system,sans-serif;max-height:500px;overflow-y:auto;border:1px solid #e0e0e0;border-radius:8px;">
  <table style="width:100%;border-collapse:collapse;font-size:14px;">
    <thead style="background:#f5f5f5;position:sticky;top:0;">
      <tr><th style="padding:8px 10px;text-align:left;">#</th>
          <th style="padding:8px 10px;">品質</th>
          <th style="padding:8px 10px;text-align:left;">時間軸</th>
          <th style="padding:8px 10px;text-align:left;">內容</th></tr>
    </thead>
    <tbody>{"".join(rows)}</tbody>
  </table>
</div>
<p style="color:#888;font-size:12px;margin-top:8px">品質指標：🟢 高　🟡 中　🔴 低（基於 avg_logprob）</p>
'''
display(HTML(html))


## STEP 6：匯出與下載


In [ ]:
#@title  6-1 一鍵產出全部格式 { display-mode: "form" }
#@markdown 產出 **TXT / SRT / VTT / JSON / TSV** 五種格式

import json

def srt_ts(s):
    ms = int(s * 1000)
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    sec, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{sec:02d},{ms:03d}"

def vtt_ts(s):
    ms = int(s * 1000)
    h, ms = divmod(ms, 3600000)
    m, ms = divmod(ms, 60000)
    sec, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{sec:02d}.{ms:03d}"

def keep(seg):
    if not CONFIG['filter_low_quality']:
        return True
    return seg.get("avg_logprob", 0) > -1.0 and seg.get("no_speech_prob", 0) < 0.6

segs = [s for s in result["segments"] if keep(s)]
n_filtered = len(result["segments"]) - len(segs)

# TXT
with open(f"{base_name}.txt", "w", encoding="utf-8") as f:
    f.write(result["text"].strip())

# SRT
with open(f"{base_name}.srt", "w", encoding="utf-8") as f:
    for i, seg in enumerate(segs, 1):
        f.write(f"{i}\n{srt_ts(seg['start'])} --> {srt_ts(seg['end'])}\n{seg['text'].strip()}\n\n")

# VTT
with open(f"{base_name}.vtt", "w", encoding="utf-8") as f:
    f.write("WEBVTT\n\n")
    for seg in segs:
        f.write(f"{vtt_ts(seg['start'])} --> {vtt_ts(seg['end'])}\n{seg['text'].strip()}\n\n")

# JSON
with open(f"{base_name}.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

# TSV
with open(f"{base_name}.tsv", "w", encoding="utf-8") as f:
    f.write("start\tend\ttext\n")
    for seg in segs:
        f.write(f"{seg['start']:.2f}\t{seg['end']:.2f}\t{seg['text'].strip()}\n")

print("匯出檔案清單")
print("━" * 40)
for ext in ["txt", "srt", "vtt", "json", "tsv"]:
    path = f"{base_name}.{ext}"
    size_kb = os.path.getsize(path) / 1024
    print(f" {path}  ({size_kb:.1f} KB)")
print("━" * 40)
if CONFIG['filter_low_quality'] and n_filtered > 0:
    print(f"已過濾 {n_filtered} 個低品質片段")


In [ ]:
#@title 6-2 下載到本機 { display-mode: "form" }
#@markdown 勾選想下載的格式

dl_txt = True   #@param {type:"boolean"}
dl_srt = True   #@param {type:"boolean"}
dl_vtt = False  #@param {type:"boolean"}
dl_json = False #@param {type:"boolean"}
dl_tsv = False  #@param {type:"boolean"}

from google.colab import files

picks = [(ext, want) for ext, want in
         [("txt", dl_txt), ("srt", dl_srt), ("vtt", dl_vtt),
          ("json", dl_json), ("tsv", dl_tsv)] if want]

if not picks:
    print("ℹ未勾選任何格式")
else:
    for ext, _ in picks:
        files.download(f"{base_name}.{ext}")
    print(f"已觸發下載：{', '.join(e for e,_ in picks)}")


## 儲存到 Google Drive

將所有輸出檔案自動同步到 Drive，方便日後查閱。


In [ ]:
#@title 同步到 Google Drive { display-mode: "form" }
#@markdown 將輸出儲存至 Drive 中的 `WhisperAI_Output/{影片名稱}/` 資料夾

from google.colab import drive
import shutil

drive.mount('/content/drive', force_remount=False)

target_dir = f"/content/drive/MyDrive/WhisperAI_Output/{base_name}"
os.makedirs(target_dir, exist_ok=True)

for ext in ["txt", "srt", "vtt", "json", "tsv"]:
    src = f"{base_name}.{ext}"
    if os.path.exists(src):
        shutil.copy(src, target_dir)
# 也備份音訊檔
if audio_file and os.path.exists(audio_file):
    shutil.copy(audio_file, target_dir)

print(f"已備份至：{target_dir}")
print(f"開啟：https://drive.google.com/drive/my-drive")


---
## 批次處理多個 YouTube 影片

可重複執行此格，將多個 URL 依序辨識並輸出。


In [ ]:
#@title 批次辨識（選用） { display-mode: "form" }
#@markdown **多個 YouTube URL，以逗號或換行分隔**（單行表單支援逗號分隔；如需多行，請直接在程式碼中修改 `batch_urls`）

batch_urls = "" #@param {type:"string"}

import os, re, yt_dlp, whisper

# 內嵌依賴函式（避免順序依賴）
def _safe_name(s, n=60):
    s = re.sub(r'[\\/*?:"<>|]', "_", s).strip()
    return s[:n] if s else "output"

def _srt_ts(s):
    ms = int(s * 1000)
    h, ms = divmod(ms, 3600000); m, ms = divmod(ms, 60000); sec, ms = divmod(ms, 1000)
    return f"{h:02d}:{m:02d}:{sec:02d},{ms:03d}"

# 解析：支援逗號或換行分隔
urls = [u.strip() for u in re.split(r"[,\n]+", batch_urls) if u.strip()]

if not urls:
    print("尚未填入任何 URL。可用逗號分隔多個網址，或在此 cell 直接修改 batch_urls 變數。")
else:
    print(f"共 {len(urls)} 個影片待處理\n")
    # 重用既有模型，無則重新載入
    try:
        _ = model
    except NameError:
        print(f"載入模型：{CONFIG['model_size']}")
        model = whisper.load_model(CONFIG['model_size'])

    for idx, url in enumerate(urls, 1):
        print(f"━━━━ [{idx}/{len(urls)}] {url} ━━━━")
        tmp_audio = f"batch_{idx}.m4a"
        try:
            with yt_dlp.YoutubeDL({"quiet": True, "skip_download": True, "no_warnings": True}) as ydl:
                info = ydl.extract_info(url, download=False)
            name = _safe_name(info.get("title", f"batch_{idx}"))
            opts = {
                "format": "bestaudio/best",
                "outtmpl": f"batch_{idx}.%(ext)s",
                "overwrites": True,
                "postprocessors": [{"key": "FFmpegExtractAudio",
                                    "preferredcodec": "m4a", "preferredquality": "192"}],
                "quiet": True, "no_warnings": True,
            }
            with yt_dlp.YoutubeDL(opts) as ydl:
                ydl.download([url])

            tr_opts = dict(task=CONFIG['task'], beam_size=CONFIG['beam_size'], verbose=False)
            if CONFIG['language'] != "auto":
                tr_opts['language'] = CONFIG['language']
            r = model.transcribe(tmp_audio, **tr_opts)

            with open(f"{name}.txt", "w", encoding="utf-8") as f:
                f.write(r['text'].strip())
            with open(f"{name}.srt", "w", encoding="utf-8") as f:
                for i, seg in enumerate(r['segments'], 1):
                    f.write(f"{i}\n{_srt_ts(seg['start'])} --> {_srt_ts(seg['end'])}\n{seg['text'].strip()}\n\n")
            print(f"完成 → {name}.txt / {name}.srt")
        except Exception as e:
            print(f"失敗：{e}")
        finally:
            # 清理暫存音檔
            if os.path.exists(tmp_audio):
                try: os.remove(tmp_audio)
                except OSError: pass
    print("\n批次處理完成")
